In [2]:
import sys
sys.path.append('..')

In [3]:
from src.data.load import load_dataset
from src.data.split import temporal_split, random_split
import glob, os, pandas as pd
from src.config import ID_FOLDER

dataset_v2 = load_dataset()

csv_paths = glob.glob(os.path.join(ID_FOLDER, "*", "*_clean.csv"))
dfs = []
for p in csv_paths:
    year = os.path.basename(os.path.dirname(p))
    df_year = pd.read_csv(p, low_memory=False)
    df_year["year_folder"] = year
    dfs.append(df_year)
isolates_v2 = pd.concat(dfs, ignore_index=True)

split_temporal = temporal_split(dataset_v2, isolates_v2)
split_random = random_split(dataset_v2)

print("n_samples:", dataset_v2.n_samples)
print("Train:", split_temporal['train_idx'].shape, "Test:", split_temporal['test_idx'].shape)

temporal_split: train=3331, test=1313
random_split: train=3483, test=1161
n_samples: 4644
Train: (3331,) Test: (1313,)


In [4]:
spectrum = dataset_v2.X[0]

print("shape:", spectrum.shape)
print("dtype:", spectrum.dtype)
print("n_peaks:", spectrum.n_peaks)

print("\nintensities shape:", spectrum.intensities.shape)
print("intensities sample:", spectrum.intensities[:10])

print("\nmass_to_charge_ratios shape:", spectrum.mass_to_charge_ratios.shape)
print("mass_to_charge_ratios sample:", spectrum.mass_to_charge_ratios[:10])

shape: (6000, 2)
dtype: float64
n_peaks: 6000

intensities shape: (6000,)
intensities sample: [0.00034792 0.00048636 0.00013401 0.00072679 0.0012926  0.00101188
 0.00021698 0.00017965 0.00014184 0.00030657]

mass_to_charge_ratios shape: (6000,)
mass_to_charge_ratios sample: [0. 1. 2. 3. 4. 5. 6. 7. 8. 9.]


In [5]:
import importlib
from src.data import features
importlib.reload(features)
from src.data.features import to_feature_matrix

X = to_feature_matrix(dataset_v2)
print("X shape:", X.shape)
print("X dtype:", X.dtype)
print("Sample row (first 10 values):", X[0, :10])

X shape: (4644, 6000)
X dtype: float64
Sample row (first 10 values): [0.00034792 0.00048636 0.00013401 0.00072679 0.0012926  0.00101188
 0.00021698 0.00017965 0.00014184 0.00030657]


In [6]:
import pandas as pd
from src.data.load import load_dataset, load_metadata_with_years

dataset_v3 = load_dataset()
metadata_years = load_metadata_with_years()

# Build a dataframe: code, year_folder, and each antibiotic's label
merged = dataset_v3.y.merge(metadata_years, on='code', how='left')

print("R rate per year, per antibiotic:\n")
for ab in ['Ciprofloxacin', 'Cotrimoxazole', 'Ceftriaxone', 
           'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin']:
    print(f"\n{ab}:")
    print(merged.groupby('year_folder')[ab].mean().round(3))

R rate per year, per antibiotic:


Ciprofloxacin:
year_folder
2015    0.315
2016    0.317
2017    0.284
2018    0.273
Name: Ciprofloxacin, dtype: object

Cotrimoxazole:
year_folder
2015    0.393
2016    0.336
2017    0.322
2018    0.316
Name: Cotrimoxazole, dtype: object

Ceftriaxone:
year_folder
2015    0.225
2016    0.241
2017    0.204
2018    0.183
Name: Ceftriaxone, dtype: object

Amoxicillin-Clavulanic acid:
year_folder
2015    0.337
2016    0.257
2017     0.21
2018    0.285
Name: Amoxicillin-Clavulanic acid, dtype: object

Ampicillin-Amoxicillin:
year_folder
2015    0.573
2016    0.589
2017    0.573
2018    0.589
Name: Ampicillin-Amoxicillin, dtype: object


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from src.data.features import to_feature_matrix

X = to_feature_matrix(dataset_v3)

# Binary label: 1 if 2018, 0 if 2015-2017
years = merged['year_folder'].astype(int)
y_year = (years == 2018).astype(int).to_numpy()

print("Class balance (fraction 2018):", y_year.mean())

X_train, X_test, y_train, y_test = train_test_split(
    X, y_year, test_size=0.25, random_state=42, stratify=y_year
)

model = XGBClassifier(random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
auroc = roc_auc_score(y_test, y_proba)
print(f"\nYear-classifier AUROC (2018 vs 2015-2017): {auroc:.4f}")
print("(AUROC near 0.5 = no detectable spectral shift; near 1.0 = strong shift)")

Class balance (fraction 2018): 0.2827304048234281

Year-classifier AUROC (2018 vs 2015-2017): 0.9571
(AUROC near 0.5 = no detectable spectral shift; near 1.0 = strong shift)


In [8]:
import numpy as np
from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.config import RANDOM_SEED

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

OPTION_A = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
            'Ciprofloxacin', 'Cotrimoxazole']
OPTION_D = ['Ceftriaxone', 'Ciprofloxacin', 'Ampicillin-Amoxicillin',
            'Cotrimoxazole', 'Amoxicillin-Clavulanic acid']

for order, name in [(OPTION_A, 'Option A'), (OPTION_D, 'Option D')]:
    Y = np.column_stack([dataset.to_numpy(ab) for ab in order])
    X_train, X_test = X[split['train_idx']], X[split['test_idx']]
    Y_train, Y_test = Y[split['train_idx']], Y[split['test_idx']]

    chain = ClassifierChain(
        estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
        order=list(range(len(order))),
        random_state=RANDOM_SEED,
    )
    chain.fit(X_train, Y_train)
    Y_pred = chain.predict(X_test)

    all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
    all_zero_true_frac = (Y_test.sum(axis=1) == 0).mean()
    print(f"{name}: fraction all-zero predictions = {all_zero_frac:.4f} "
          f"(true all-zero fraction = {all_zero_true_frac:.4f})")

temporal_split: train=3331, test=1313
Option A: fraction all-zero predictions = 0.3899 (true all-zero fraction = 0.3641)
Option D: fraction all-zero predictions = 0.6535 (true all-zero fraction = 0.3641)


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split
from src.evaluation.metrics import multilabel_metrics
from src.config import RANDOM_SEED

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

order = ['Ciprofloxacin', 'Ceftriaxone', 'Ampicillin-Amoxicillin',
         'Cotrimoxazole', 'Amoxicillin-Clavulanic acid']

Y = np.column_stack([dataset.to_numpy(ab) for ab in order])

# Flip Ciprofloxacin's labels (column 0) — R becomes majority, everything else unchanged
print("Original Ciprofloxacin R rate:", Y[:, 0].mean())
Y[:, 0] = 1 - Y[:, 0]
print("Flipped Ciprofloxacin R rate:", Y[:, 0].mean())

train_idx, test_idx = split['train_idx'], split['test_idx']
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

Y_pred = chain.predict(X_test)
Y_proba = chain.predict_proba(X_test)

for i, ab in enumerate(order):
    label = f"{ab} (FLIPPED)" if i == 0 else ab
    auroc = roc_auc_score(Y_test[:, i], Y_proba[:, i])
    f1 = f1_score(Y_test[:, i], Y_pred[:, i])
    print(f"{label}: AUROC={auroc:.4f}, F1={f1:.4f}")

joint_metrics = multilabel_metrics(Y_test, Y_pred)
all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
print(f"\njoint metrics: {joint_metrics}")
print(f"all_zero_frac: {all_zero_frac:.4f}")

temporal_split: train=3331, test=1313
Original Ciprofloxacin R rate: 0.2904823428079242
Flipped Ciprofloxacin R rate: 0.7095176571920758


KeyboardInterrupt: 

In [ ]:
order_f = ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin',
           'Cotrimoxazole', 'Ampicillin-Amoxicillin']

Y = np.column_stack([dataset.to_numpy(ab) for ab in order_f])

X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(order_f))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

Y_pred = chain.predict(X_test)
Y_proba = chain.predict_proba(X_test)

for i, ab in enumerate(order_f):
    auroc = roc_auc_score(Y_test[:, i], Y_proba[:, i])
    f1 = f1_score(Y_test[:, i], Y_pred[:, i])
    print(f"{ab}: AUROC={auroc:.4f}, F1={f1:.4f}")

joint_metrics = multilabel_metrics(Y_test, Y_pred)
all_zero_frac = (Y_pred.sum(axis=1) == 0).mean()
print(f"\njoint metrics: {joint_metrics}")
print(f"all_zero_frac: {all_zero_frac:.4f}")

Ceftriaxone: AUROC=0.8617, F1=0.5539
Amoxicillin-Clavulanic acid: AUROC=0.5951, F1=0.1042
Ciprofloxacin: AUROC=0.7585, F1=0.4256
Cotrimoxazole: AUROC=0.6512, F1=0.2192
Ampicillin-Amoxicillin: AUROC=0.6065, F1=0.3480

joint metrics: {'hamming_loss': 0.27844630616907845, 'jaccard_score': 0.09058136582889058}
all_zero_frac: 0.8378


c:\Admin - Vaishali\Academics\VITV\Project_4_1\Implementation\amr_env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Jaccard is ill-defined and being set to 0.0 in samples with no true or predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
option_a_random = pd.read_csv('../results/metrics/chains_random_seeds.csv')
option_a_zero = option_a_random[option_a_random['order_name'] == 'option_a']['all_zero_frac']
print("Option A all_zero_frac across 10 seeds:", option_a_zero.tolist())
print("Mean:", option_a_zero.mean(), "Std:", option_a_zero.std())

Option A all_zero_frac across 10 seeds: [0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3267326732673267, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.3061690784463061, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.2741812642802742, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3533891850723534, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3313023610053313, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.3259710586443259, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.2962680883472963, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3297791317593297, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.3556740289413557, 0.2901751713632902, 0.2901751713632902, 0.2901751713632902,

In [ ]:
order = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
         'Ciprofloxacin', 'Cotrimoxazole']

Y_full = np.column_stack([dataset.to_numpy(ab) for ab in order])
Y_test_true = Y_full[test_idx]

true_all_zero_frac = (Y_test_true.sum(axis=1) == 0).mean()
print(f"True all-zero fraction (temporal test set): {true_all_zero_frac:.4f}")
print(f"n={len(Y_test_true)}, all-zero count={int((Y_test_true.sum(axis=1) == 0).sum())}")

True all-zero fraction (temporal test set): 0.3641
n=1313, all-zero count=478


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

# quick synthetic check: does predict_proba return columns in original Y order,
# regardless of the chain's internal `order`?
X_dummy = np.random.rand(50, 5)
Y_dummy = np.random.randint(0, 2, size=(50, 3))  # 3 labels, easy to distinguish

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[2, 0, 1], random_state=0)
chain.fit(X_dummy, Y_dummy)

print("chain.order_:", chain.order_)  # the actual fitting order used
pred = chain.predict(X_dummy[:5])
print("predict() output shape:", pred.shape)  # should be (5, 3) - matching Y_dummy's original column count/order

chain.order_: [2, 0, 1]
predict() output shape: (5, 3)


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier

X_dummy = np.random.rand(200, 5)

# Make 3 labels that are EASY to tell apart and trivially learnable from X_dummy
Y_dummy = np.column_stack([
    (X_dummy[:, 0] > 0.5).astype(int),   # label 0: depends only on X col 0
    (X_dummy[:, 1] > 0.5).astype(int),   # label 1: depends only on X col 1
    (X_dummy[:, 2] > 0.5).astype(int),   # label 2: depends only on X col 2
])

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[2, 0, 1], random_state=0)
chain.fit(X_dummy, Y_dummy)

pred = chain.predict(X_dummy)

# If column order is correct, predicted column i should closely match Y_dummy column i
for i in range(3):
    accuracy = (pred[:, i] == Y_dummy[:, i]).mean()
    print(f"Column {i}: accuracy against Y_dummy column {i} = {accuracy:.3f}")

Column 0: accuracy against Y_dummy column 0 = 1.000
Column 1: accuracy against Y_dummy column 1 = 1.000
Column 2: accuracy against Y_dummy column 2 = 1.000


In [ ]:
# Quick check: is the temporal Option A vs baseline gap comparable to random-split noise?

baseline_temporal_jaccard = 0.2323
option_a_temporal_jaccard = 0.2453
temporal_gap = option_a_temporal_jaccard - baseline_temporal_jaccard

# reference noise magnitude from random split (10 seeds each)
baseline_random_std = 0.0047   # from earlier: 0.273 ± 0.005
option_a_random_std = 0.0063   # from earlier: 0.291 ± 0.006
pooled_std = (baseline_random_std**2 + option_a_random_std**2) ** 0.5

print(f"Temporal gap: {temporal_gap:.4f}")
print(f"Reference pooled std (from random-split seed variance): {pooled_std:.4f}")
print(f"Gap as multiple of reference std: {temporal_gap / pooled_std:.2f}")

Temporal gap: 0.0130
Reference pooled std (from random-split seed variance): 0.0079
Gap as multiple of reference std: 1.65


In [ ]:
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
import numpy as np

X_dummy = np.random.rand(20, 5)
Y_dummy = np.random.randint(0, 2, size=(20, 3))

chain = ClassifierChain(estimator=XGBClassifier(eval_metric='logloss'), order=[0, 1, 2], random_state=0)
chain.fit(X_dummy, Y_dummy)

# Inspect the actual estimators inside the chain
for i, est in enumerate(chain.estimators_):
    print(f"Step {i}: n_features expected = {est.n_features_in_}")

Step 0: n_features expected = 5
Step 1: n_features expected = 6
Step 2: n_features expected = 7


In [ ]:
import sys
sys.path.append('..')

from src.data.load import load_dataset, load_metadata_with_years
from src.data.features import to_feature_matrix
from src.data.split import temporal_split

dataset = load_dataset()
X = to_feature_matrix(dataset)
metadata_df = load_metadata_with_years()
split = temporal_split(dataset, metadata_df)

print("n_samples:", dataset.n_samples)
print("X shape:", X.shape)

temporal_split: train=3331, test=1313
n_samples: 4644
X shape: (4644, 6000)


In [ ]:
import numpy as np
from sklearn.multioutput import ClassifierChain
from xgboost import XGBClassifier
from src.config import RANDOM_SEED

OPTION_A = ['Ampicillin-Amoxicillin', 'Ceftriaxone', 'Amoxicillin-Clavulanic acid',
            'Ciprofloxacin', 'Cotrimoxazole']

Y = np.column_stack([dataset.to_numpy(ab) for ab in OPTION_A])
X_train, X_test = X[split['train_idx']], X[split['test_idx']]
Y_train = Y[split['train_idx']]

chain = ClassifierChain(
    estimator=XGBClassifier(random_state=RANDOM_SEED, eval_metric='logloss'),
    order=list(range(len(OPTION_A))),
    random_state=RANDOM_SEED,
)
chain.fit(X_train, Y_train)

# small subsample for a quick check
X_sub = X_test[:50]

# manual recursion, same logic as shap_chain
prior_pred_columns = []
for i in range(len(OPTION_A)):
    aug_sub = np.hstack([X_sub] + prior_pred_columns) if prior_pred_columns else X_sub
    pred_i = chain.estimators_[i].predict(aug_sub)
    prior_pred_columns.append(pred_i.reshape(-1, 1))

manual_preds = np.hstack(prior_pred_columns)  # shape (50, 5), in chain order (0..4)

# official chain.predict on same subsample - note this returns columns in 
# ORIGINAL Y order (order=[0,1,2,3,4] here so it's identical to chain order anyway)
official_preds = chain.predict(X_sub)

print("Manual recursion matches chain.predict() exactly:", np.array_equal(manual_preds, official_preds))

Manual recursion matches chain.predict() exactly: True


In [ ]:
import numpy as np

baseline_full = np.load('../results/metrics/shap_baseline_temporal_full.npz')
chain_full = np.load('../results/metrics/shap_chain_option_a_temporal_full.npz')

for ab in ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'Cotrimoxazole']:
    baseline_top15 = set(np.argsort(baseline_full[ab])[::-1][:15])
    chain_top15 = set(np.argsort(chain_full[ab])[::-1][:15])
    overlap = len(baseline_top15 & chain_top15)
    print(f"{ab}: {overlap}/15 top spectral bins overlap between baseline and chain")

Ceftriaxone: 10/15 top spectral bins overlap between baseline and chain
Amoxicillin-Clavulanic acid: 7/15 top spectral bins overlap between baseline and chain
Ciprofloxacin: 6/15 top spectral bins overlap between baseline and chain
Cotrimoxazole: 2/15 top spectral bins overlap between baseline and chain


In [ ]:
import numpy as np
from xgboost import XGBClassifier
import shap

X_train, X_test = X[split['train_idx']], X[split['test_idx']]
X_sub = X_test[:750]  # same subsample size as before, doesn't need to match exact indices for this noise check

antibiotics_to_check = ['Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'Cotrimoxazole']

for ab in antibiotics_to_check:
    y = dataset.to_numpy(ab)
    y_train = y[split['train_idx']]

    model_seed1 = XGBClassifier(random_state=42, eval_metric='logloss')
    model_seed1.fit(X_train, y_train)
    shap1 = shap.TreeExplainer(model_seed1).shap_values(X_sub)
    if isinstance(shap1, list):
        shap1 = shap1[1] if len(shap1) > 1 else shap1[0]
    top15_seed1 = set(np.argsort(np.abs(shap1).mean(axis=0))[::-1][:15])

    model_seed2 = XGBClassifier(random_state=123, eval_metric='logloss')
    model_seed2.fit(X_train, y_train)
    shap2 = shap.TreeExplainer(model_seed2).shap_values(X_sub)
    if isinstance(shap2, list):
        shap2 = shap2[1] if len(shap2) > 1 else shap2[0]
    top15_seed2 = set(np.argsort(np.abs(shap2).mean(axis=0))[::-1][:15])

    overlap = len(top15_seed1 & top15_seed2)
    print(f"{ab}: {overlap}/15 top-15 overlap between two seeds (baseline, same data)")

Ceftriaxone: 15/15 top-15 overlap between two seeds (baseline, same data)
Amoxicillin-Clavulanic acid: 15/15 top-15 overlap between two seeds (baseline, same data)
Ciprofloxacin: 15/15 top-15 overlap between two seeds (baseline, same data)
Cotrimoxazole: 15/15 top-15 overlap between two seeds (baseline, same data)


In [ ]:
order_g = ['Ampicillin-Amoxicillin', 'Cotrimoxazole', 'Ciprofloxacin',
           'Amoxicillin-Clavulanic acid', 'Ceftriaxone']

from src.interpretability.shap_analysis import shap_chain
chain_g_results = shap_chain(X, dataset, split, 'temporal', order_g, 'option_g', subsample_n=750)

[shap_chain:option_g] fitting chain...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): building augmented input...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): running TreeExplainer on (750, 6000)...
[shap_chain:option_g] step 0 (Ampicillin-Amoxicillin): fraction_prior_attribution=0.0000
[shap_chain:option_g] step 1 (Cotrimoxazole): building augmented input...
[shap_chain:option_g] step 1 (Cotrimoxazole): running TreeExplainer on (750, 6001)...
[shap_chain:option_g] step 1 (Cotrimoxazole): fraction_prior_attribution=0.1040
[shap_chain:option_g] step 2 (Ciprofloxacin): building augmented input...
[shap_chain:option_g] step 2 (Ciprofloxacin): running TreeExplainer on (750, 6002)...
[shap_chain:option_g] step 2 (Ciprofloxacin): fraction_prior_attribution=0.0923
[shap_chain:option_g] step 3 (Amoxicillin-Clavulanic acid): building augmented input...
[shap_chain:option_g] step 3 (Amoxicillin-Clavulanic acid): running TreeExplainer on (750, 6003)...
[shap_chain:option_g] step

In [ ]:
baseline_full = np.load('../results/metrics/shap_baseline_temporal_full.npz')

for ab in ['Cotrimoxazole', 'Ciprofloxacin', 'Amoxicillin-Clavulanic acid', 'Ceftriaxone']:
    baseline_top15 = set(np.argsort(baseline_full[ab])[::-1][:15])
    chain_g_top15 = set(chain_g_results[ab]['top15_spectral_feature_indices'])
    overlap = len(baseline_top15 & chain_g_top15)
    position = chain_g_results[ab]['chain_position']
    print(f"{ab} (position {position}): {overlap}/15 top spectral bins overlap with baseline")

Cotrimoxazole (position 1): 3/15 top spectral bins overlap with baseline
Ciprofloxacin (position 2): 8/15 top spectral bins overlap with baseline
Amoxicillin-Clavulanic acid (position 3): 6/15 top spectral bins overlap with baseline
Ceftriaxone (position 4): 9/15 top spectral bins overlap with baseline


In [11]:
from src.data.load import load_metadata_with_years
import glob, os
import pandas as pd
from src.config import ID_FOLDER

# load_metadata_with_years() only returns code + year_folder (trimmed),
# but we need the FULL metadata with all antibiotic columns here,
# so rebuild it directly the same way load_metadata_with_years does internally
paths = glob.glob(os.path.join(ID_FOLDER, '*', '*_clean.csv'))
frames = []
for path in paths:
    year_folder = os.path.basename(os.path.dirname(path))
    df = pd.read_csv(path, low_memory=False)
    df['year_folder'] = year_folder
    frames.append(df)
metadata = pd.concat(frames, ignore_index=True)

print("metadata shape:", metadata.shape)

metadata shape: (111257, 93)


In [12]:
import pandas as pd

target_species = ['Staphylococcus aureus', 'Klebsiella pneumoniae', 'Pseudomonas aeruginosa']

# Reuse the combined metadata (all years) we already have as `metadata` from earlier exploration,
# or rebuild it via load_metadata_with_years-style loading if not in memory
non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

for species in target_species:
    print(f"\n{'='*60}")
    print(f"SPECIES: {species}")
    print('='*60)

    species_df = metadata[metadata['species'] == species]
    print(f"Total isolates: {len(species_df)}")

    for ab in antibiotic_cols:
        col = species_df[ab]
        non_missing = col[(col.notna()) & (col != '-')]
        if len(non_missing) < 200:  # skip antibiotics with too little data to matter
            continue

        # collapse to binary using same rule as labels.py: any R or I -> 1, pure S -> 0
        def to_binary(v):
            if 'R' in v: return 1
            if 'I' in v: return 1
            if 'S' in v: return 0
            return None

        binary = non_missing.apply(to_binary).dropna()
        if len(binary) < 200:
            continue

        r_rate = binary.mean()
        majority = "R-majority" if r_rate > 0.5 else "S-majority"
        print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f} ({majority})")


SPECIES: Staphylococcus aureus
Total isolates: 6994
  Piperacillin-Tazobactam: n=3640, R-rate=0.196 (S-majority)
  Meropenem: n=3643, R-rate=0.195 (S-majority)
  Ciprofloxacin: n=3789, R-rate=0.171 (S-majority)
  Cefepime: n=3640, R-rate=0.196 (S-majority)
  Cotrimoxazole: n=3771, R-rate=0.045 (S-majority)
  Imipenem: n=3640, R-rate=0.196 (S-majority)
  Ceftriaxone: n=3640, R-rate=0.196 (S-majority)
  Clindamycin: n=3635, R-rate=0.159 (S-majority)
  Amoxicillin-Clavulanic acid: n=3640, R-rate=0.196 (S-majority)
  Vancomycin: n=3791, R-rate=0.000 (S-majority)
  Penicillin: n=3634, R-rate=0.741 (R-majority)
  Erythromycin: n=3640, R-rate=0.190 (S-majority)
  Tetracycline: n=3643, R-rate=0.084 (S-majority)
  Ampicillin-Amoxicillin: n=3637, R-rate=0.741 (R-majority)
  Linezolid: n=3639, R-rate=0.000 (S-majority)
  Teicoplanin: n=3631, R-rate=0.002 (S-majority)
  Tigecycline: n=3640, R-rate=0.000 (S-majority)
  Daptomycin: n=3783, R-rate=0.009 (S-majority)
  Gentamicin: n=3643, R-rate=0.03

In [13]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

pen = species_df['Penicillin']
amp = species_df['Ampicillin-Amoxicillin']

# only compare where both are non-missing
both_present = (pen.notna()) & (pen != '-') & (amp.notna()) & (amp != '-')
print(f"Isolates with both non-missing: {both_present.sum()}")

pen_valid = pen[both_present]
amp_valid = amp[both_present]

exact_match = (pen_valid == amp_valid).mean()
print(f"Exact string match rate: {exact_match:.4f}")

# also check after collapsing to binary (R/I->1, S->0), since exact string 
# match is stricter than binary match (e.g. 'R(1), S(1)' vs 'R' would differ 
# as strings but both collapse to 1)
def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

pen_binary = pen_valid.apply(to_binary)
amp_binary = amp_valid.apply(to_binary)

binary_match = (pen_binary == amp_binary).mean()
print(f"Binary (R/I=1, S=0) match rate: {binary_match:.4f}")

Isolates with both non-missing: 3634
Exact string match rate: 1.0000
Binary (R/I=1, S=0) match rate: 1.0000


In [14]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']
non_antibiotic_cols = ['code', 'species', 'laboratory_species', 'year_folder',
                        'Unnamed: 0.1', 'Unnamed: 0']
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

print("Near-balanced labels (40-55% R) for S. aureus:\n")
for ab in antibiotic_cols:
    col = species_df[ab]
    non_missing = col[(col.notna()) & (col != '-')]
    if len(non_missing) < 200:
        continue
    binary = non_missing.apply(to_binary).dropna()
    if len(binary) < 200:
        continue
    r_rate = binary.mean()
    if 0.40 <= r_rate <= 0.55:
        print(f"  {ab}: n={len(binary)}, R-rate={r_rate:.3f}")

print("\n(if nothing printed above, no near-balanced labels exist for S. aureus)")

Near-balanced labels (40-55% R) for S. aureus:


(if nothing printed above, no near-balanced labels exist for S. aureus)


In [15]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

candidates = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem', 
              'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 
              'Oxacillin', 'Cefazolin']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

print("Pairwise binary match rates among suspiciously-identical beta-lactams:\n")
for i in range(len(candidates)):
    for j in range(i+1, len(candidates)):
        a, b = candidates[i], candidates[j]
        col_a, col_b = species_df[a], species_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        if match_rate > 0.95:
            print(f"  {a} <-> {b}: n={both_present.sum()}, match={match_rate:.4f}  *** LIKELY DUPLICATE ***")

Pairwise binary match rates among suspiciously-identical beta-lactams:

  Piperacillin-Tazobactam <-> Meropenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefepime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Imipenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Ceftriaxone: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Amoxicillin-Clavulanic acid: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefuroxime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Oxacillin: n=3640, match=0.9975  *** LIKELY DUPLICATE ***
  Piperacillin-Tazobactam <-> Cefazolin: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Cefepime: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Imipenem: n=3640, match=1.0000  *** LIKELY DUPLICATE ***
  Meropenem <-> Ceftriaxone: n=3640, match=1.0000  *** LIKE

In [16]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

cluster_reps = ['Oxacillin', 'Piperacillin-Tazobactam', 'Cefazolin']
for rep in cluster_reps:
    col_pen, col_rep = species_df['Penicillin'], species_df[rep]
    both_present = (col_pen.notna()) & (col_pen != '-') & (col_rep.notna()) & (col_rep != '-')
    bin_pen = col_pen[both_present].apply(to_binary)
    bin_rep = col_rep[both_present].apply(to_binary)
    match_rate = (bin_pen == bin_rep).mean()
    print(f"Penicillin <-> {rep}: n={both_present.sum()}, match={match_rate:.4f}")

Penicillin <-> Oxacillin: n=3634, match=0.4576
Penicillin <-> Piperacillin-Tazobactam: n=3634, match=0.4551
Penicillin <-> Cefazolin: n=3634, match=0.4551


In [17]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

col_clin, col_ery = species_df['Clindamycin'], species_df['Erythromycin']
both_present = (col_clin.notna()) & (col_clin != '-') & (col_ery.notna()) & (col_ery != '-')
bin_clin = col_clin[both_present].apply(to_binary)
bin_ery = col_ery[both_present].apply(to_binary)
match_rate = (bin_clin == bin_ery).mean()
print(f"Clindamycin <-> Erythromycin: n={both_present.sum()}, match={match_rate:.4f}")

Clindamycin <-> Erythromycin: n=3631, match=0.9317


In [18]:
cluster = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem', 
           'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 
           'Oxacillin', 'Cefazolin']

print("Non-missing counts per antibiotic in cluster:")
for ab in cluster:
    col = species_df[ab]
    n = ((col.notna()) & (col != '-')).sum()
    print(f"  {ab}: n={n}")

# check if the SAME isolates (by code) are tested across all 9
masks = []
for ab in cluster:
    col = species_df[ab]
    masks.append((col.notna()) & (col != '-'))

all_same_mask = masks[0]
for m in masks[1:]:
    all_same_mask = all_same_mask & (m == masks[0])  # do all masks match mask[0] exactly, per-row?
print(f"\nRows where ALL 9 antibiotics share the exact same missingness pattern as Piperacillin-Tazobactam: {all_same_mask.sum()} / {len(species_df)}")

# quantify Oxacillin's disagreements specifically - how many isolates, which years
col_pip, col_oxa = species_df['Piperacillin-Tazobactam'], species_df['Oxacillin']
both = (col_pip.notna()) & (col_pip != '-') & (col_oxa.notna()) & (col_oxa != '-')
bin_pip = col_pip[both].apply(to_binary)
bin_oxa = col_oxa[both].apply(to_binary)
disagree_mask = bin_pip != bin_oxa
print(f"\nPiperacillin-Tazobactam vs Oxacillin disagreements: {disagree_mask.sum()} isolates")
if disagree_mask.sum() > 0:
    disagree_years = species_df.loc[both][disagree_mask.values]['year_folder']
    print("Disagreements by year:")
    print(disagree_years.value_counts().sort_index())

Non-missing counts per antibiotic in cluster:
  Piperacillin-Tazobactam: n=3640
  Meropenem: n=3643
  Cefepime: n=3640
  Imipenem: n=3640
  Ceftriaxone: n=3640
  Amoxicillin-Clavulanic acid: n=3640
  Cefuroxime: n=3640
  Oxacillin: n=3791
  Cefazolin: n=3640

Rows where ALL 9 antibiotics share the exact same missingness pattern as Piperacillin-Tazobactam: 3640 / 6994

Piperacillin-Tazobactam vs Oxacillin disagreements: 9 isolates
Disagreements by year:
year_folder
2017    3
2018    6
Name: count, dtype: int64


In [19]:
ecoli_df = metadata[metadata['species'] == 'Escherichia coli']

# test the same beta-lactam cluster antibiotics in E. coli
ecoli_cluster_candidates = ['Piperacillin-Tazobactam', 'Meropenem', 'Cefepime', 'Imipenem',
                              'Ceftriaxone', 'Amoxicillin-Clavulanic acid', 'Cefuroxime', 'Cefazolin']

print("Pairwise match rates for the same antibiotic cluster, tested on E. coli:\n")
for i in range(len(ecoli_cluster_candidates)):
    for j in range(i+1, len(ecoli_cluster_candidates)):
        a, b = ecoli_cluster_candidates[i], ecoli_cluster_candidates[j]
        col_a, col_b = ecoli_df[a], ecoli_df[b]
        both_present = (col_a.notna()) & (col_a != '-') & (col_b.notna()) & (col_b != '-')
        if both_present.sum() < 200:
            continue
        bin_a = col_a[both_present].apply(to_binary)
        bin_b = col_b[both_present].apply(to_binary)
        match_rate = (bin_a == bin_b).mean()
        flag = "  *** HIGH MATCH ***" if match_rate > 0.95 else ""
        print(f"  {a} <-> {b}: n={both_present.sum()}, match={match_rate:.4f}{flag}")

Pairwise match rates for the same antibiotic cluster, tested on E. coli:

  Piperacillin-Tazobactam <-> Meropenem: n=4863, match=0.9149
  Piperacillin-Tazobactam <-> Cefepime: n=4870, match=0.8078
  Piperacillin-Tazobactam <-> Imipenem: n=4862, match=0.9157
  Piperacillin-Tazobactam <-> Ceftriaxone: n=4870, match=0.7768
  Piperacillin-Tazobactam <-> Amoxicillin-Clavulanic acid: n=4865, match=0.8183
  Meropenem <-> Cefepime: n=4934, match=0.8121
  Meropenem <-> Imipenem: n=4933, match=0.9982  *** HIGH MATCH ***
  Meropenem <-> Ceftriaxone: n=4934, match=0.7756
  Meropenem <-> Amoxicillin-Clavulanic acid: n=4927, match=0.7327
  Cefepime <-> Imipenem: n=4934, match=0.8121
  Cefepime <-> Ceftriaxone: n=4986, match=0.9531  *** HIGH MATCH ***
  Cefepime <-> Amoxicillin-Clavulanic acid: n=4977, match=0.7555
  Imipenem <-> Ceftriaxone: n=4934, match=0.7762
  Imipenem <-> Amoxicillin-Clavulanic acid: n=4927, match=0.7333
  Ceftriaxone <-> Amoxicillin-Clavulanic acid: n=4979, match=0.7499


In [20]:
species_df = metadata[metadata['species'] == 'Staphylococcus aureus']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

# Check Penicillin's own availability first, since it's the non-negotiable seed
pen_col = species_df['Penicillin']
pen_available = (pen_col.notna()) & (pen_col != '-')
print(f"Penicillin availability: {pen_available.sum()} / {len(species_df)}")

core_four = ['Penicillin', 'Oxacillin', 'Ciprofloxacin', 'Cotrimoxazole']

panels = {
    'core4_plus_clindamycin': core_four + ['Clindamycin'],
    'core4_plus_erythromycin': core_four + ['Erythromycin'],
    'core4_plus_both': core_four + ['Clindamycin', 'Erythromycin'],
}

for panel_name, antibiotics in panels.items():
    mask = pd.Series(True, index=species_df.index)
    for ab in antibiotics:
        col = species_df[ab]
        mask &= (col.notna()) & (col != '-')
    print(f"\n{panel_name} ({len(antibiotics)} antibiotics): complete cases = {mask.sum()}")

    # per-year breakdown, since we'll need this for temporal split later
    year_counts = species_df.loc[mask, 'year_folder'].value_counts().sort_index()
    print(f"  Per-year: {dict(year_counts)}")

Penicillin availability: 3634 / 6994

core4_plus_clindamycin (5 antibiotics): complete cases = 3626
  Per-year: {'2015': np.int64(67), '2016': np.int64(1035), '2017': np.int64(1380), '2018': np.int64(1144)}

core4_plus_erythromycin (5 antibiotics): complete cases = 3632
  Per-year: {'2015': np.int64(67), '2016': np.int64(1036), '2017': np.int64(1381), '2018': np.int64(1148)}

core4_plus_both (6 antibiotics): complete cases = 3626
  Per-year: {'2015': np.int64(67), '2016': np.int64(1035), '2017': np.int64(1380), '2018': np.int64(1144)}


In [21]:
panel = ['Penicillin', 'Oxacillin', 'Ciprofloxacin', 'Cotrimoxazole', 'Clindamycin', 'Erythromycin']

def to_binary(v):
    if 'R' in v: return 1
    if 'I' in v: return 1
    if 'S' in v: return 0
    return None

mask = pd.Series(True, index=species_df.index)
for ab in panel:
    col = species_df[ab]
    mask &= (col.notna()) & (col != '-')

complete_df = species_df[mask].copy()
print("Complete-case count:", len(complete_df))

labels_sa = complete_df[panel].apply(lambda col: col.apply(to_binary))

print("\nClass balance (proportion R) per antibiotic:")
print(labels_sa.mean().round(4))

print("\n6x6 correlation matrix:")
print(labels_sa.corr().round(3))

Complete-case count: 3626

Class balance (proportion R) per antibiotic:
Penicillin       0.7413
Oxacillin        0.1980
Ciprofloxacin    0.1616
Cotrimoxazole    0.0339
Clindamycin      0.1586
Erythromycin     0.1900
dtype: float64

6x6 correlation matrix:
               Penicillin  Oxacillin  Ciprofloxacin  Cotrimoxazole  \
Penicillin          1.000      0.294          0.145          0.086   
Oxacillin           0.294      1.000          0.353          0.148   
Ciprofloxacin       0.145      0.353          1.000          0.129   
Cotrimoxazole       0.086      0.148          0.129          1.000   
Clindamycin         0.067      0.192          0.168          0.044   
Erythromycin        0.082      0.248          0.234          0.049   

               Clindamycin  Erythromycin  
Penicillin           0.067         0.082  
Oxacillin            0.192         0.248  
Ciprofloxacin        0.168         0.234  
Cotrimoxazole        0.044         0.049  
Clindamycin          1.000         0.7

In [22]:
corr_matrix = labels_sa.corr().abs()
for ab in panel:
    mean_corr = corr_matrix[ab].drop(ab).mean()
    print(f"{ab}: mean |correlation| with others = {mean_corr:.4f}")

Penicillin: mean |correlation| with others = 0.1347
Oxacillin: mean |correlation| with others = 0.2468
Ciprofloxacin: mean |correlation| with others = 0.2059
Cotrimoxazole: mean |correlation| with others = 0.0911
Clindamycin: mean |correlation| with others = 0.2476
Erythromycin: mean |correlation| with others = 0.2762
